# 01b Fetch with Business (Strategy C: hybrid DOM + heuristics)

AAPL / MSFT / GOOGL / NVDA / TSLA / AVGO / AMAT / AMZN / META の
10-K × 5年 + 10-Q × 15四半期から、以下のセクションを抽出して階層的にチャンク化する。

| form | Part    | Item    | `section_key` | `section_role` |
| ---- | ------- | ------- | ------------- | -------------- |
| 10-K | Part I  | Item 1  | `item_1`      | `business`     |
| 10-K | Part I  | Item 1A | `item_1a`     | `risk_factors` |
| 10-K | Part II | Item 7  | `item_7`      | `mda`          |
| 10-Q | Part II | Item 1A | `item_1a`     | `risk_factors` |
| 10-Q | Part I  | Item 2  | `item_2`      | `mda`          |

## 戦略 C: ハイブリッド DOM + ヒューリスティック

```
Filing
  └ Item (edgartools API)
      ├ Text 抽出      ← get_item_with_part(part, item) ── 9/9 銘柄で動く
      ├ Table 除外     ← doc.sections[<key>].tables() ── 40/45 filings で DOM-level
      │                   失敗時は _is_table_like ヒューリスティックに fallback
      └ Subsection 検出 ← 強シグナル正規表現 + Title Case 判定 (04 から移植)
          └ Paragraph packing ← 段落単位で max_tokens まで詰める
              └ Chunk (gte-Qwen2 tokenizer, max=1024)
```

## 設計判断と参照

- **`section_key` は form 内 Item 番号そのまま** (10-K の MD&A は `item_7`、10-Q は `item_2`)
- **`section_role` は意味タグ** (`business` / `risk_factors` / `mda`) — form 横断集計用
- **MAX_TOKENS = 1024** (04 の 512 から増、gte-Qwen2 32K 上限の中で実用最適)
- **tokenizer = `Alibaba-NLP/gte-Qwen2-1.5B-instruct`** (後段 embedding と一致)
- **DOM section lookup は 2 命名規則を試す** (`part_i_item_1` と `Item 1`)
- 出力は `data/*_v2.parquet`。既存 `chunks.parquet` / `chunks_hier.parquet` には触れない
- **`_helpers.py` には依存しない**
- 移植元: `notebook/FILING_NLP/04_hierarchical_chunking.ipynb`


In [ ]:
# Cell 1: 自己完結セットアップ (HF_HOME, edgar identity, パス, 警告フィルタ)
import logging
import os
import re
from pathlib import Path

import torch

PKG_DIR = Path.cwd()
DATA_DIR = PKG_DIR / "data"
HF_CACHE_DIR = DATA_DIR / "hf_cache"
FILINGS_V2_PARQUET = DATA_DIR / "filings_v2.parquet"
SECTIONS_V2_PARQUET = DATA_DIR / "sections_v2.parquet"
CHUNKS_V2_PARQUET = DATA_DIR / "chunks_v2.parquet"

DATA_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# EDGAR identity を .env から読む
import edgar  # noqa: E402

identity = os.environ.get("EDGAR_IDENTITY")
if not identity:
    env_path = PKG_DIR.parent.parent / ".env"
    if env_path.exists():
        for line in env_path.read_text().splitlines():
            line = line.strip()
            if line.startswith("EDGAR_IDENTITY="):
                identity = line.split("=", 1)[1].strip().strip('"').strip("'")
                os.environ["EDGAR_IDENTITY"] = identity
                break
if not identity:
    raise RuntimeError(
        "EDGAR_IDENTITY を .env に 'EDGAR_IDENTITY=Name email@example.com' で設定"
    )
edgar.set_identity(identity)

# torch device
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


# edgartools の legacy parser 警告抑制
class _LegacyParserFilter(logging.Filter):
    def filter(self, record):
        return "falling back to legacy parser" not in record.getMessage()


_edgar_logger = logging.getLogger("edgar.core")
if not any(isinstance(f, _LegacyParserFilter) for f in _edgar_logger.filters):
    _edgar_logger.addFilter(_LegacyParserFilter())

print("device:", device)
print("DATA_DIR:", DATA_DIR)
print("identity:", identity)


device: mps
DATA_DIR: /Users/yukihata/Desktop/quants/notebook/FILING_NLP/data
identity: YH-05 youxitiancore@gmail.com


In [2]:
# Cell 2: 10-K × 5年 + 10-Q × 15四半期を取得
from concurrent.futures import ThreadPoolExecutor

from tqdm.auto import tqdm

TICKERS = ["AAPL", "MSFT", "GOOGL", "NVDA", "TSLA", "AVGO", "AMAT", "AMZN", "META"]


def _fetch(ticker: str, form: str, limit: int):
    try:
        return ticker, list(edgar.Company(ticker).get_filings(form=form).head(limit))
    except Exception as e:  # noqa: BLE001
        return ticker, e


with ThreadPoolExecutor(max_workers=3) as ex:
    filings_10k = dict(ex.map(lambda t: _fetch(t, "10-K", 5), TICKERS))
print("===== 10-K =====")
for t, r in filings_10k.items():
    print(t, len(r) if not isinstance(r, Exception) else f"ERR: {r}")

with ThreadPoolExecutor(max_workers=3) as ex:
    filings_10q = dict(ex.map(lambda t: _fetch(t, "10-Q", 15), TICKERS))
print("===== 10-Q =====")
for t, r in filings_10q.items():
    print(t, len(r) if not isinstance(r, Exception) else f"ERR: {r}")


===== 10-K =====
AAPL 5
MSFT 5
GOOGL 5
NVDA 5
TSLA 5
AVGO 5
AMAT 5
AMZN 5
META 5
===== 10-Q =====
AAPL 15
MSFT 15
GOOGL 15
NVDA 15
TSLA 15
AVGO 15
AMAT 15
AMZN 15
META 15


In [ ]:
# Cell 3: filings_v2.parquet 保存
import pandas as pd

all_filing_objs = []
for d, label in [(filings_10k, "10-K"), (filings_10q, "10-Q")]:
    for ticker, result in d.items():
        if isinstance(result, Exception):
            continue
        for f in result:
            all_filing_objs.append((ticker, label, f))

df_filings = (
    pd.DataFrame(
        [
            {
                "filing_id": str(f.accession_number),
                "ticker": ticker,
                "form": form,
                "filing_date": pd.Timestamp(str(f.filing_date)),
                "accession_number": str(f.accession_number),
            }
            for ticker, form, f in all_filing_objs
        ]
    )
    .sort_values(["ticker", "form", "filing_date"])
    .reset_index(drop=True)
)
df_filings.to_parquet(FILINGS_V2_PARQUET)
print("saved:", FILINGS_V2_PARQUET, "rows:", len(df_filings))
df_filings.groupby(["form", "ticker"]).size().unstack(fill_value=0)


saved: /Users/yukihata/Desktop/quants/notebook/FILING_NLP/data/filings_v2.parquet rows: 180


ticker,AAPL,AMAT,AMZN,AVGO,GOOGL,META,MSFT,NVDA,TSLA
form,,,,,,,,,
10-K,5,5,5,5,5,5,5,5,5
10-Q,15,15,15,15,15,15,15,15,15


## セクション抽出 + DOM-level Table 除外

### text 抽出

`obj.get_item_with_part(part, item, markdown=False)` で plain text を取得。9/9 銘柄で動作。
`markdown=True` は調査の結果 plain text と byte 単位で同一の出力なので使わない。

### DOM section lookup

`obj.document.sections` は 2 種類の命名規則が混在する:

- snake_case (`part_i_item_1`) — AAPL, NVDA, AVGO, META
- 生 Item ラベル (`Item 1`) — MSFT, GOOGL, AMZN, AMAT

両方を試す `get_section()` ヘルパーで 9/9 銘柄の 40/45 filing をカバー。
失敗ケース (TSLA の一部) では DOM table 除外を skip し、ヒューリスティック判定のみ使う。

### Table 除外戦略

1. **DOM-level**: section から `sec.tables()` で TableNode を取得し、各 table の text を
   元 section text から削除
2. **safety net (ヒューリスティック)**: 段落 packing 後に `_is_table_like()` で残存
   テーブル断片 (`(In millions, except percentages)202220212020` 等) を除外


In [ ]:
# Cell 4: form 別 ITEM_SPECS + get_section ヘルパー

# (section_key, section_role, part, item) — 04 から拡張 (Item 1 を 10-K に追加)
ITEM_SPECS_10K: list[tuple[str, str, str, str]] = [
    ("item_1", "business", "Part I", "Item 1"),
    ("item_1a", "risk_factors", "Part I", "Item 1A"),
    ("item_7", "mda", "Part II", "Item 7"),
]
ITEM_SPECS_10Q: list[tuple[str, str, str, str]] = [
    ("item_1a", "risk_factors", "Part II", "Item 1A"),
    ("item_2", "mda", "Part I", "Item 2"),
]
ITEM_SPECS_BY_FORM: dict[str, list[tuple[str, str, str, str]]] = {
    "10-K": ITEM_SPECS_10K,
    "10-Q": ITEM_SPECS_10Q,
}


def get_section(doc, part: str, item: str):
    """edgartools の 2 命名規則を吸収する section lookup.

    候補:
        1. f"{part_normalized}_{item_normalized}" — "part_i_item_1a"
        2. item — "Item 1A"
        3. item.upper() — "ITEM 1A"
    どれもヒットしなければ None.
    """
    part_norm = part.lower().replace(" ", "_")
    item_norm = item.lower().replace(" ", "_")
    candidates = [
        f"{part_norm}_{item_norm}",
        item,
        item.upper(),
    ]
    for key in candidates:
        sec = doc.sections.get(key)
        if sec is not None:
            return sec, key
    return None, None


for form, specs in ITEM_SPECS_BY_FORM.items():
    print(f"{form}:")
    for sk, sr, p, i in specs:
        print(f"  {sk:8s} ({sr:13s}) <- {p} / {i}")


10-K:
  item_1   (business     ) <- Part I / Item 1
  item_1a  (risk_factors ) <- Part I / Item 1A
  item_7   (mda          ) <- Part II / Item 7
10-Q:
  item_1a  (risk_factors ) <- Part II / Item 1A
  item_2   (mda          ) <- Part I / Item 2


In [ ]:
# Cell 5: TableNode から text を再構成 + section text から削除するヘルパー


def table_to_text(table) -> str:
    """TableNode から検索用テキストを再構成.

    edgartools の TableNode は rows: list[Row], 各 Row は cells: list[Cell] を持つ。
    Cell.content が実セル文字列。table.text や table.content があれば優先利用。
    """
    # API バリエーション吸収
    if hasattr(table, "text") and isinstance(table.text, str):
        return table.text
    if hasattr(table, "content") and isinstance(table.content, str):
        return table.content
    # rows から再構成
    rows = getattr(table, "rows", [])
    headers = getattr(table, "headers", [])
    parts = []
    for h in headers:
        cells = getattr(h, "cells", None) or [h]
        parts.append(" ".join((getattr(c, "content", "") or "").strip() for c in cells))
    for r in rows:
        cells = getattr(r, "cells", []) or []
        parts.append(" ".join((getattr(c, "content", "") or "").strip() for c in cells))
    return "\n".join(p for p in parts if p)


def remove_tables_from_text(section_text: str, tables) -> tuple[str, int]:
    """section text から table 由来テキストを削除. (cleaned_text, removed_count) を返す.

    マッチ戦略 (段階的):
        1. table.text 完全一致 (rare だが clean)
        2. 連続空白を正規化したうえで部分一致 → 該当箇所を改行2個に置換
        3. マッチしない table は無視 (heuristic safety net に任せる)
    """
    if not tables:
        return section_text, 0

    cleaned = section_text
    removed = 0

    def _norm(s: str) -> str:
        # 連続空白 (nbsp 含む) を 1 空白に
        return re.sub(r"\s+", " ", s.replace("\xa0", " ")).strip()

    cleaned_norm_index = None  # 必要なら遅延構築

    for t in tables:
        try:
            txt = table_to_text(t)
        except Exception:
            continue
        txt = (txt or "").strip()
        if len(txt) < 20:  # ノイズ
            continue

        # 戦略 1: 完全一致
        if txt in cleaned:
            cleaned = cleaned.replace(txt, "\n\n[TABLE REMOVED]\n\n", 1)
            removed += 1
            continue

        # 戦略 2: 正規化マッチ
        # cleaned_norm を一度だけ作る
        if cleaned_norm_index is None:
            cleaned_norm = _norm(cleaned)
            cleaned_norm_index = cleaned_norm
        norm_txt = _norm(txt)
        if len(norm_txt) >= 30 and norm_txt in cleaned_norm_index:
            # 正規化マッチした場合、最初の数語 (40 文字程度) を anchor として
            # cleaned 内で検索する。これはおおよその除去でしかないが、
            # safety net の _is_table_like で残骸を catch する想定。
            anchor = norm_txt[:60]
            # cleaned の "正規化" を文字単位で行う代わりに、anchor の先頭単語列で粗 match
            anchor_words = anchor.split()[:5]
            if anchor_words:
                pat = re.escape(" ".join(anchor_words)).replace(r"\ ", r"\s+")
                m = re.search(pat, cleaned)
                if m:
                    # 該当行〜段落末を削除
                    start = m.start()
                    # 段落末 (\n\n) まで
                    end_match = cleaned.find("\n\n", start)
                    end = (
                        end_match
                        if end_match >= 0
                        else min(start + len(txt) + 100, len(cleaned))
                    )
                    cleaned = (
                        cleaned[:start] + "\n\n[TABLE REMOVED]\n\n" + cleaned[end:]
                    )
                    removed += 1
                    cleaned_norm_index = None  # 再計算が必要
                    continue

    return cleaned, removed


In [ ]:
# Cell 6: 全 filing × Item で text 抽出 + DOM table 除外
section_rows: list[dict] = []
miss: list[tuple[str, str, str, str]] = []  # (ticker, fid, form, section_key)
dom_stats: list[dict] = []

for ticker, form, f in tqdm(all_filing_objs, desc="extracting sections"):
    specs = ITEM_SPECS_BY_FORM.get(form)
    if specs is None:
        continue
    try:
        obj = f.obj()
    except Exception as e:  # noqa: BLE001
        print(f"obj() fail: {ticker} {form} {f.accession_number}: {e}")
        for sk, _, _, _ in specs:
            miss.append((ticker, str(f.accession_number), form, sk))
        continue

    fid = str(f.accession_number)
    try:
        doc = obj.document
    except Exception:
        doc = None

    for section_key, section_role, part, item in specs:
        # text 抽出 (全銘柄で動く primary path)
        try:
            text = obj.get_item_with_part(part, item, markdown=False)
        except Exception as e:  # noqa: BLE001
            print(f"  get_item_with_part fail: {ticker} {fid} {part}/{item}: {e}")
            text = None
        if not isinstance(text, str) or not text.strip():
            miss.append((ticker, fid, form, section_key))
            continue

        original_len = len(text)
        n_tables_removed = 0
        section_key_resolved = None

        # DOM table 除外 (best-effort)
        if doc is not None:
            sec, sec_key = get_section(doc, part, item)
            section_key_resolved = sec_key
            if sec is not None:
                try:
                    tbls = sec.tables() if callable(sec.tables) else sec.tables
                except Exception:
                    tbls = None
                if tbls:
                    text, n_tables_removed = remove_tables_from_text(text, tbls)

        dom_stats.append(
            {
                "ticker": ticker,
                "form": form,
                "filing_id": fid,
                "section_key": section_key,
                "dom_section_found": section_key_resolved is not None,
                "dom_section_key": section_key_resolved,
                "tables_removed": n_tables_removed,
                "len_before": original_len,
                "len_after": len(text),
            }
        )

        section_rows.append(
            {
                "filing_id": fid,
                "ticker": ticker,
                "form": form,
                "section_key": section_key,
                "section_role": section_role,
                "part": part,
                "item": item,
                "text": text,
                "char_count": len(text),
            }
        )

print(f"\nextracted: {len(section_rows)} sections, miss: {len(miss)}")


extracting sections:   0%|          | 0/180 [00:00<?, ?it/s]


extracted: 399 sections, miss: 6


In [7]:
# Cell 7: sections_v2.parquet 保存 + DOM 除外統計
df_sections = pd.DataFrame(section_rows)
df_sections.to_parquet(SECTIONS_V2_PARQUET)
print("saved:", SECTIONS_V2_PARQUET, "rows:", len(df_sections))

# DOM 除外統計
df_dom = pd.DataFrame(dom_stats)
print("\n=== DOM table 除外サマリ ===")
print(f"  section 取得成功 (DOM): {df_dom['dom_section_found'].sum()} / {len(df_dom)}")
print(f"  table 除外件数 平均:    {df_dom['tables_removed'].mean():.2f} per section")
print(
    f"  text 削減 平均:        {(df_dom['len_before'] - df_dom['len_after']).mean():.0f} chars"
)
print(
    f"  text 削減率 平均:       {((df_dom['len_before'] - df_dom['len_after']) / df_dom['len_before'].clip(lower=1)).mean():.1%}"
)

print("\n=== ticker × section_key 別 平均 tables_removed ===")
print(
    df_dom.pivot_table(
        index="ticker", columns="section_key", values="tables_removed", aggfunc="mean"
    ).round(1)
)

print("\n=== miss (section 取得失敗) ===")
df_miss = pd.DataFrame(miss, columns=["ticker", "filing_id", "form", "section_key"])
print(df_miss)


saved: /Users/yukihata/Desktop/quants/notebook/FILING_NLP/data/sections_v2.parquet rows: 399

=== DOM table 除外サマリ ===
  section 取得成功 (DOM): 336 / 399
  table 除外件数 平均:    1.30 per section
  text 削減 平均:        353 chars
  text 削減率 平均:       0.5%

=== ticker × section_key 別 平均 tables_removed ===
section_key  item_1  item_1a  item_2  item_7
ticker                                      
AAPL            0.0      0.0     0.0     0.0
AMAT            0.6      0.0     0.0     0.0
AMZN            0.0      0.0     0.0     0.0
AVGO            0.0      0.0     0.0     0.0
GOOGL           1.2      0.3     0.5     1.6
META            0.0      0.0     0.0     0.0
MSFT           21.2      1.6    11.7    23.2
NVDA            0.0      0.0     0.0     0.0
TSLA            0.3      0.0     3.7     0.0

=== miss (section 取得失敗) ===
  ticker             filing_id  form section_key
0   TSLA  0001104659-26-053166  10-K      item_1
1   TSLA  0001104659-26-053166  10-K     item_1a
2   TSLA  0001104659-26-053166  10-

## Subsection 検出 (04 のヒューリスティック + Business 拡張)

### 強シグナル正規表現

- Risk Factors 系: `Risks Related to`, `Risks Associated with`, `Strategic Risks`, ...
- MD&A 系: `Overview`, `Results of Operations`, `Liquidity and Capital Resources`, ...
- **Item 1 Business 系 (新規追加)**: `Products`, `Services`, `Markets and Distribution`,
  `Manufacturing`, `Supply of Components`, `Research and Development`,
  `Intellectual Property`, `Human Capital`, `Government Regulation`, `Competition`,
  `Available Information`, `Operating Segments`, ...

### 弱シグナル (Title Case ヒューリスティック)

- 行長 5-120 字
- 末尾が句読点でない
- 数字/bullet で開始しない
- Title Case 様 (単語の 60%+ が大文字始まり)

### ノイズ除去

- **Page artifact**: `Apple Inc. | 2025 Form 10-K | 5`, `Page 12`, `Annual Report` 等
- **Bullet 段落化**: `\n•` を `\n\n•` に置換 (段落区切り昇格)

### 出力

各 section を `[(subsection_title, subsection_text), ...]` のリストに分解。
小見出しが見つからない場合は section 全体を 1 subsection として扱う。


In [ ]:
# Cell 8: 強シグナル / page artifact / table marker の正規表現群 (04 + Business 拡張)
STRONG_HEADING_PATTERNS = [
    # --- Risk Factors 系 (04 既存) ---
    r"^Risks?\s+Related\s+to\s+",
    r"^Risks?\s+Relating\s+to\s+",
    r"^Risks?\s+Associated\s+with\s+",
    r"^(Strategic|Operational|Financial|Legal|Regulatory|Market|Macroeconomic|"
    r"Industry|Business|General|Cybersecurity|Tax|Intellectual\s+Property|"
    r"Human\s+Capital|Compliance|Environmental|Climate|Geopolitical|"
    r"Legal\s+and\s+Regulatory\s+Compliance)\s+Risks?\b",
    # --- MD&A 系 (04 既存) ---
    r"^Overview\b",
    r"^Results\s+of\s+Operations\b",
    r"^Liquidity\s+and\s+Capital\s+Resources\b",
    r"^Critical\s+Accounting\s+(Estimates|Policies|Judgments)\b",
    r"^Recent\s+Accounting\s+Pronouncements\b",
    r"^Off-Balance\s+Sheet\s+Arrangements\b",
    r"^Contractual\s+Obligations\b",
    r"^Foreign\s+Currency\b",
    r"^Segment\s+(Results|Operating\s+Performance|Information)\b",
    # --- Item 1 Business 系 (v2 新規) ---
    r"^Products?\b",
    r"^Services\b",
    r"^Markets?\s+and\s+Distribution\b",
    r"^Manufacturing\b",
    r"^Supply\s+(of\s+Components|Chain)\b",
    r"^Research\s+and\s+Development\b",
    r"^Patents,?\s+Trademarks?\b",
    r"^Intellectual\s+Property\b",
    r"^Human\s+Capital(\s+Resources)?\b",
    r"^Employees\b",
    r"^Government\s+Regulation\b",
    r"^Competition\b",
    r"^(Business\s+)?Seasonality\b",
    r"^Available\s+Information\b",
    r"^Corporate\s+(Information|History)\b",
    r"^Operating\s+Segments?\b",
    r"^(Business\s+)?Strategy\b",
    r"^Company\s+Background\b",
    r"^Our\s+Company\b",
    r"^General\b",  # MSFT は "GENERAL" を使う
    r"^What\s+We\s+Offer\b",  # MSFT
]
STRONG_HEADING_RE = re.compile("|".join(STRONG_HEADING_PATTERNS), re.IGNORECASE)

PAGE_ARTIFACT_PATTERNS = [
    r"\|",
    r"\bForm\s+(?:10|8|11|20|S)-?\s*[KQABFN]?\b",
    r"\bPage\s+\d+\b",
    r"\bAnnual\s+Report\b",
    r"\bQuarterly\s+Report\b",
    r"\bTable\s+of\s+Contents\b",
]
PAGE_ARTIFACT_RE = re.compile("|".join(PAGE_ARTIFACT_PATTERNS), re.IGNORECASE)

# table marker (safety net 用)
TABLE_MARKER_RE = re.compile(
    r"(Three|Six|Nine|Twelve)\s+Months\s+Ended|"
    r"Year(s)?\s+Ended\s+(December|January|June|September|October|November)|"
    r"Fiscal\s+Year\s+Ended|"
    r"Percentage\s*Change|"
    r"\(In\s+millions|"
    r"\(In\s+thousands",
    re.IGNORECASE,
)


def _normalize(s: str) -> str:
    return re.sub(r"\s+", " ", s.replace("\xa0", " ")).strip()


def _is_page_artifact(line: str) -> bool:
    return bool(PAGE_ARTIFACT_RE.search(_normalize(line)))


def _is_table_like(text: str) -> bool:
    """safety net: DOM 除外を通過した残骸テーブルを catch."""
    s = _normalize(text)
    # 短くて数字密度高い (ヘッダー行 残骸: '(In millions, except percentages)202220212020')
    digit_ratio = sum(c.isdigit() for c in s) / max(len(s), 1)
    if len(s) < 100 and digit_ratio > 0.2:
        return True
    if TABLE_MARKER_RE.search(text):
        multi_space_count = len(re.findall(r" {3,}", text))
        if digit_ratio > 0.05 or multi_space_count > 5:
            return True
    # 除外マーカー (本セル内で挿入)
    if "[TABLE REMOVED]" in text:
        return True
    return False


def _looks_like_heading(line: str) -> bool:
    s = _normalize(line)
    if not (5 <= len(s) <= 120):
        return False
    if s[-1] in ".,;:":
        return False
    if s[0].isdigit() or s[0] in ("•", "-", "*"):
        return False
    if re.match(r"^\([a-z]\)|^\([0-9]+\)|^[ivx]+\.", s, re.IGNORECASE):
        return False
    if re.search(r"\s\d{1,4}\s*$", s):
        return False
    if _is_page_artifact(s):
        return False
    words = re.findall(r"[A-Za-z]+", s)
    if not words:
        return False
    cap = sum(1 for w in words if w[0].isupper())
    return cap / len(words) >= 0.6


def _preprocess(text: str) -> str:
    """bullet 行を段落区切りに昇格."""
    text = re.sub(r"\n[•·]", "\n\n•", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text


# 単体テスト
_test_cases = [
    ("Apple Inc. | 2025 Form 10-K | 5", True),
    ("Microsoft Corporation Page 12", True),
    ("Macroeconomic and Industry Risks", False),
    ("Products", False),
    ("Human Capital", False),
]
for _text, _expected in _test_cases:
    assert _is_page_artifact(_text) == _expected, (
        f"_is_page_artifact({_text!r}) expected {_expected}"
    )

_strong_cases = [
    ("Risks Related to Our Operations", True),
    ("Results of Operations", True),
    ("Products", True),
    ("Human Capital", True),
    ("Intellectual Property", True),
    ("Just some random text.", False),
]
for _text, _expected in _strong_cases:
    matched = bool(STRONG_HEADING_RE.match(_text))
    assert matched == _expected, (
        f"STRONG_HEADING_RE({_text!r}) expected {_expected}, got {matched}"
    )

print(f"_is_page_artifact: {len(_test_cases)} tests passed")
print(f"STRONG_HEADING_RE: {len(_strong_cases)} tests passed")


In [ ]:
# Cell 9: split_subsections (04 のまま移植)
def split_subsections(text: str) -> list[tuple[str, str]]:
    """(subsection_title, subsection_text) のリストに分割.

    1. _preprocess で bullet を段落区切りに昇格
    2. 連続改行で block 分割
    3. 単一行 block が page artifact → スキップ
    4. block の先頭行が strong/weak heading → 新 subsection 開始
    5. 小見出しゼロなら [("", text)] を返す
    """
    text = _preprocess(text.strip())
    blocks = text.split("\n\n")

    items: list[tuple[str, str]] = []
    cur_title = ""
    cur_body: list[str] = []

    def flush() -> None:
        if cur_body:
            items.append((cur_title, "\n\n".join(cur_body).strip()))

    for block in blocks:
        b = block.strip()
        if not b:
            continue
        lines = b.split("\n")
        first_line = lines[0].strip()
        # 単一行 block
        if len(lines) == 1:
            if _is_page_artifact(first_line):
                continue
            if STRONG_HEADING_RE.match(first_line) or _looks_like_heading(first_line):
                flush()
                cur_title = first_line
                cur_body = []
                continue
        # 複数行 block: 先頭が strong のみ採用
        elif STRONG_HEADING_RE.match(first_line) and not _is_page_artifact(first_line):
            flush()
            cur_title = first_line
            cur_body = []
            rest = "\n".join(lines[1:]).strip()
            if rest:
                cur_body.append(rest)
            continue
        cur_body.append(b)
    flush()

    if not items:
        return [("", text)]
    return items


In [ ]:
# Cell 10: 全 section に subsection 分割を適用
all_subs_rows = []
for row in df_sections.to_dict("records"):
    subs = split_subsections(row["text"])
    for sub_idx, (title, body) in enumerate(subs):
        all_subs_rows.append(
            {
                "filing_id": row["filing_id"],
                "ticker": row["ticker"],
                "form": row["form"],
                "section_key": row["section_key"],
                "section_role": row["section_role"],
                "subsection_idx": sub_idx,
                "subsection_title": title,
                "text": body,
                "char_count": len(body),
            }
        )
df_subs = pd.DataFrame(all_subs_rows)
print(f"total subsections: {len(df_subs)}")
print("\n=== subsections per (form, section_key) ===")
print(df_subs.groupby(["form", "section_key"]).size().to_frame("count"))
print("\n=== subsections per filing × item の分布 ===")
print(df_subs.groupby(["filing_id", "section_key"]).size().describe().round(2))
print("\n=== 出現頻度トップ subsection titles (空除く) ===")
print(
    df_subs[df_subs["subsection_title"] != ""]["subsection_title"]
    .value_counts()
    .head(20)
)


## 段落 packing + チャンク化

### 設定

- **tokenizer**: `Alibaba-NLP/gte-Qwen2-1.5B-instruct` (後段 embedding と一致)
- **MAX_TOKENS = 1024** (gte-Qwen2 の理論上限 32K の中で実用最適)
- 段落単位で詰める。単一段落が 1024 超 → 文単位で強制分割
- 段落 packing 後に `_is_table_like()` を safety net として適用 (残存テーブル断片を除外)

### 上限超過 chunk の対処

04 と同じく、単一文が MAX_TOKENS 超なら諦めて 1 chunk として吐く (Cell 14 で詳細調査)。


In [ ]:
# Cell 11: gte-Qwen2 tokenizer ロード
from transformers import AutoTokenizer

EMBED_MODEL_ID = "Alibaba-NLP/gte-Qwen2-1.5B-instruct"
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_ID, trust_remote_code=True)
print("tokenizer:", type(tokenizer).__name__, "vocab:", tokenizer.vocab_size)


In [ ]:
# Cell 12: 段落 packing (04 のロジックを MAX_TOKENS=1024 で)
MAX_TOKENS = 1024


def _sentence_split(text: str) -> list[str]:
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z])", text)
    return [p.strip() for p in parts if p.strip()]


def _tok_count(s: str) -> int:
    return len(tokenizer.encode(s, add_special_tokens=False))


def paragraph_pack(text: str, max_tokens: int = MAX_TOKENS) -> list[str]:
    text = text.strip()
    if not text:
        return []
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    chunks: list[str] = []
    cur: list[str] = []
    cur_tok = 0
    for p in paragraphs:
        n = _tok_count(p)
        if n > max_tokens:
            if cur:
                chunks.append("\n\n".join(cur))
                cur, cur_tok = [], 0
            sentences = _sentence_split(p)
            sub_cur: list[str] = []
            sub_tok = 0
            for s in sentences:
                sn = _tok_count(s)
                if sub_tok + sn > max_tokens and sub_cur:
                    chunks.append(" ".join(sub_cur))
                    sub_cur, sub_tok = [], 0
                sub_cur.append(s)
                sub_tok += sn
            if sub_cur:
                chunks.append(" ".join(sub_cur))
            continue
        if cur_tok + n > max_tokens and cur:
            chunks.append("\n\n".join(cur))
            cur, cur_tok = [], 0
        cur.append(p)
        cur_tok += n
    if cur:
        chunks.append("\n\n".join(cur))
    return chunks


# 動作確認
sample = df_subs.iloc[0]
sample_chunks = paragraph_pack(sample["text"])
print(
    f"sample: {sample['ticker']} {sample['form']} "
    f"{sample['section_key']} sub[{sample['subsection_idx']}] "
    f"title={sample['subsection_title']!r}"
)
print(f"  char_count={sample['char_count']} -> {len(sample_chunks)} chunks")
for i, c in enumerate(sample_chunks[:3]):
    print(f"  chunk[{i}] {_tok_count(c):5d} tokens | head: {c[:100]!r}")


In [ ]:
# Cell 13: 全 subsection に packing 適用 + _is_table_like で safety net + 保存
chunk_rows = []
table_skipped = 0
for row in tqdm(df_subs.to_dict("records"), desc="chunking"):
    chunks = paragraph_pack(row["text"])
    for i, c in enumerate(chunks):
        if _is_table_like(c):
            table_skipped += 1
            continue
        chunk_rows.append(
            {
                "filing_id": row["filing_id"],
                "ticker": row["ticker"],
                "form": row["form"],
                "section_key": row["section_key"],
                "section_role": row["section_role"],
                "subsection_idx": row["subsection_idx"],
                "subsection_title": row["subsection_title"],
                "chunk_idx": i,
                "text": c,
                "token_count": _tok_count(c),
            }
        )

df_chunks = pd.DataFrame(chunk_rows).merge(
    df_filings[["filing_id", "filing_date"]], on="filing_id", how="left"
)
df_chunks.to_parquet(CHUNKS_V2_PARQUET)
print(f"saved: {CHUNKS_V2_PARQUET}")
print(f"total chunks: {len(df_chunks)}")
print(f"safety-net で除外された table chunk: {table_skipped}")


## 検証セル (04 から移植 + v2 拡張)

確認ポイント:

1. **短い subsection 保存**: 合計トークンが MAX_TOKENS 以下の subsection は 1 chunk のまま残る
2. **上限遵守**: 全 chunk の `token_count <= MAX_TOKENS`
3. **section 別 chunk 数バランス**: ticker × form × item で過不足ないか
4. **時系列の連続性**: 同一 ticker・同一 item で年度ごとに subsection 数が大きく振れないか
5. **DOM table 除外の効果**: ticker × section_key 別の tables_removed 集計
6. **Item 1 Business の chunk 規模**: 新規追加項目の妥当性チェック


In [ ]:
# Cell 14: 健全性チェック (04 Cell 12 + v2 追加項目)
print("=== 設定 ===")
print(f"  MAX_TOKENS:           {MAX_TOKENS}")
print(f"  総 chunk 数:           {len(df_chunks):,}")
print(f"  対象 ticker 数:        {df_chunks['ticker'].nunique()}")
print(f"  対象 filing 数:        {df_chunks['filing_id'].nunique()}")

print("\n=== 上限・下限の健全性 ===")
over = int((df_chunks["token_count"] > MAX_TOKENS).sum())
empty = int((df_chunks["token_count"] == 0).sum())
tiny = int((df_chunks["token_count"] < 50).sum())
print(f"  上限超過 chunk:        {over}  (期待値: 0)")
print(f"  空 chunk:              {empty}")
print(f"  極小 chunk (<50 tok):  {tiny}")

print("\n=== subsection ごとの packing 統計 ===")
chunks_per_sub = (
    df_chunks.groupby(["filing_id", "section_key", "subsection_idx"])
    .agg(n_chunks=("chunk_idx", "size"), total_tokens=("token_count", "sum"))
    .reset_index()
)
n_total = len(chunks_per_sub)
n_single = int((chunks_per_sub["n_chunks"] == 1).sum())
print(
    f"  1 chunk で済んだ subsection: {n_single} / {n_total}  ({n_single / n_total:.1%})"
)
short_subs = chunks_per_sub[chunks_per_sub["total_tokens"] <= MAX_TOKENS]
short_split = int((short_subs["n_chunks"] > 1).sum())
print(f"  短 subsection (<={MAX_TOKENS} tok) 分割異常: {short_split} (期待値: 0)")

print("\n=== Token 数分布 ===")
print(df_chunks["token_count"].describe().round(1).to_string())

print("\n=== Packing 効率 (token_count / MAX_TOKENS) ===")
fill = df_chunks["token_count"] / MAX_TOKENS
print(f"  平均充填率: {fill.mean():.1%}, 中央値: {fill.median():.1%}")
print(f"  IQR:        {fill.quantile(0.25):.1%} - {fill.quantile(0.75):.1%}")

print("\n=== form × section_key 別 chunk 数 ===")
print(
    df_chunks.groupby(["form", "section_key", "section_role"]).size().to_frame("chunks")
)

print("\n=== ticker × section_key 別 chunk 数 ===")
print(
    df_chunks.pivot_table(
        index="ticker",
        columns=["form", "section_key"],
        values="chunk_idx",
        aggfunc="size",
        fill_value=0,
    )
)


In [ ]:
# Cell 15: 各 ticker 最新 10-K × Item 1 の subsection → chunks 対応 (04 Cell 11 拡張)
def show_chunks_per_ticker(
    ticker: str, form: str = "10-K", section_key: str = "item_1"
) -> None:
    f = (
        (df_chunks["ticker"] == ticker)
        & (df_chunks["form"] == form)
        & (df_chunks["section_key"] == section_key)
    )
    if not f.any():
        print(f"  {ticker} {form} {section_key}: データなし")
        return
    latest_fid = df_chunks[f].sort_values("filing_date").iloc[-1]["filing_id"]
    view = df_chunks[f & (df_chunks["filing_id"] == latest_fid)]
    fdate = view["filing_date"].iloc[0]
    n_subs = view["subsection_idx"].nunique()
    n_chunks = len(view)
    total_tok = int(view["token_count"].sum())
    print(
        f"【{ticker}】 {form} {fdate.date() if hasattr(fdate, 'date') else fdate} "
        f"{section_key}: {n_subs} subsections -> {n_chunks} chunks ({total_tok:,} tokens)"
    )
    by_sub = (
        view.groupby(["subsection_idx", "subsection_title"])
        .agg(
            n_chunks=("chunk_idx", "size"),
            total_tokens=("token_count", "sum"),
            max_tokens=("token_count", "max"),
        )
        .reset_index()
        .sort_values("subsection_idx")
    )
    for _, r in by_sub.iterrows():
        title = (r["subsection_title"] or "(no title)")[:55]
        tag = "[keep]" if r["n_chunks"] == 1 else f"[{r['n_chunks']:>2d}ch]"
        print(
            f"  {tag} sub[{r['subsection_idx']:2d}] {title:55} | "
            f"{r['total_tokens']:6d} tok (max {r['max_tokens']:4d})"
        )


print("=" * 80)
print("Item 1 Business")
print("=" * 80)
for t in TICKERS:
    show_chunks_per_ticker(t, "10-K", "item_1")
    print()


In [ ]:
# Cell 16: 各 ticker の最新 10-K Item 1A から最短/最長 chunk を抜粋
print("=== 各 ticker 最新 10-K Item 1A サンプル chunk ===\n")
for ticker in TICKERS:
    f = (
        (df_chunks["ticker"] == ticker)
        & (df_chunks["form"] == "10-K")
        & (df_chunks["section_key"] == "item_1a")
    )
    if not f.any():
        continue
    latest_fid = df_chunks[f].sort_values("filing_date").iloc[-1]["filing_id"]
    view = df_chunks[f & (df_chunks["filing_id"] == latest_fid)].reset_index(drop=True)
    shortest = view.loc[view["token_count"].idxmin()]
    longest = view.loc[view["token_count"].idxmax()]
    print(f"━━━ 【{ticker}】 chunks: {len(view)} ━━━")
    print(
        f"  [最短] sub[{shortest['subsection_idx']}] "
        f"{(shortest['subsection_title'] or '(no title)')!r} "
        f"chunk[{shortest['chunk_idx']}] ({shortest['token_count']} tok)"
    )
    print(f"    {shortest['text'][:300]!r}")
    print(
        f"  [最大] sub[{longest['subsection_idx']}] "
        f"{(longest['subsection_title'] or '(no title)')!r} "
        f"chunk[{longest['chunk_idx']}] ({longest['token_count']} tok)"
    )
    print(f"    {longest['text'][:300]!r}")
    print()


In [ ]:
# Cell 17: 上限超過 chunk の調査 (04 Cell 14 移植)
oversize = df_chunks[df_chunks["token_count"] > MAX_TOKENS]
n_total = len(df_chunks)
n_over = len(oversize)
print("=== 上限超過 chunk ===")
print(f"  MAX_TOKENS: {MAX_TOKENS}")
print(f"  超過件数:    {n_over} / {n_total} ({n_over / n_total:.1%})")

if n_over == 0:
    print("\n対策不要。")
else:
    print("\n=== 超過分の token 分布 ===")
    print(oversize["token_count"].describe().round(0).to_string())

    print("\n=== ticker × form × section_key 別の発生数 ===")
    print(oversize.groupby(["ticker", "form", "section_key"]).size().to_string())

    print("\n=== subsection_title 別の発生数 (top 10) ===")
    print(oversize["subsection_title"].value_counts().head(10).to_string())

    stats = pd.DataFrame(
        {
            "semicolons": oversize["text"].str.count(";"),
            "colons": oversize["text"].str.count(":"),
            "bullets": oversize["text"].str.count("•"),
            "newlines": oversize["text"].str.count("\n"),
            "periods": oversize["text"].str.count(r"\."),
        }
    )
    print("\n=== 分割ヒント ===")
    print(stats.describe().round(1).to_string())

    print("\n=== 上位 5 chunk のテキスト先頭 ===")
    top5 = oversize.nlargest(5, "token_count")
    for i, (_, r) in enumerate(top5.iterrows()):
        title = (r["subsection_title"] or "(no title)")[:40]
        print(
            f"\n--- [{i + 1}] {r['ticker']} {r['form']} sub='{title}' ({r['token_count']} tok) ---"
        )
        text_preview = r["text"][:500].replace("\n", " ⏎ ")
        print(f"  {text_preview!r}")
